# Sales Data Loading

This notebook loads all sales-related CSV files from three product line directories into Delta tables.

## Data Sources:
- **Camping**: Files/data/camping/sales/
- **Kitchen**: Files/data/kitchen/sales/
- **Ski**: Files/data/ski/sales/

## Tables to Load:
1. **Order** - Order headers from all three product lines
2. **OrderLine** - Order line items from all three product lines  
3. **OrderPayment** - Payment information from all three product lines

In [ ]:
# Setup and Configuration
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Schema Configuration
SCHEMA_NAME = "sales"
BASE_PATH = "Files/data"

# Product line paths
PRODUCT_LINES = ['camping', 'kitchen', 'ski']

# Ensure schema exists
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SCHEMA_NAME}")
print(f"✅ Schema '{SCHEMA_NAME}' ready!")
print(f"📁 Loading sales data from {len(PRODUCT_LINES)} product lines: {', '.join(PRODUCT_LINES)}")

In [ ]:
# 1. Load Order Table from All Product Lines
print("🛒 Loading Order table from all product lines...")

order_dfs = []

for product_line in PRODUCT_LINES:
    print(f"  📦 Loading {product_line} orders...")
    
    # Read CSV file
    order_df = spark.read.csv(
        f"{BASE_PATH}/{product_line}/sales/Order_Samples_{product_line.title()}.csv",
        header=True,
        inferSchema=True
    )
    
    print(f"     Records loaded: {order_df.count()}")
    order_dfs.append(order_df)

# Union all order dataframes
combined_orders_df = order_dfs[0]
for df in order_dfs[1:]:
    combined_orders_df = combined_orders_df.union(df)

total_orders = combined_orders_df.count()
print(f"\n📊 Total orders combined: {total_orders}")

# Write to Delta table
combined_orders_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_NAME}.Order")

print(f"✅ Order table loaded successfully!")

In [ ]:
# 2. Load OrderLine Table from All Product Lines
print("📝 Loading OrderLine table from all product lines...")

orderline_dfs = []

for product_line in PRODUCT_LINES:
    print(f"  📦 Loading {product_line} order lines...")
    
    # Read CSV file
    orderline_df = spark.read.csv(
        f"{BASE_PATH}/{product_line}/sales/OrderLine_Samples_{product_line.title()}.csv",
        header=True,
        inferSchema=True
    )
    
    print(f"     Records loaded: {orderline_df.count()}")
    orderline_dfs.append(orderline_df)

# Union all orderline dataframes
combined_orderlines_df = orderline_dfs[0]
for df in orderline_dfs[1:]:
    combined_orderlines_df = combined_orderlines_df.union(df)

total_orderlines = combined_orderlines_df.count()
print(f"\n📊 Total order lines combined: {total_orderlines}")

# Write to Delta table
combined_orderlines_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_NAME}.OrderLine")

print(f"✅ OrderLine table loaded successfully!")

In [ ]:
# 3. Load OrderPayment Table from All Product Lines
print("💳 Loading OrderPayment table from all product lines...")

orderpayment_dfs = []

for product_line in PRODUCT_LINES:
    print(f"  📦 Loading {product_line} order payments...")
    
    # Read CSV file
    orderpayment_df = spark.read.csv(
        f"{BASE_PATH}/{product_line}/sales/OrderPayment_{product_line.title()}.csv",
        header=True,
        inferSchema=True
    )
    
    print(f"     Records loaded: {orderpayment_df.count()}")
    orderpayment_dfs.append(orderpayment_df)

# Union all orderpayment dataframes
combined_orderpayments_df = orderpayment_dfs[0]
for df in orderpayment_dfs[1:]:
    combined_orderpayments_df = combined_orderpayments_df.union(df)

total_orderpayments = combined_orderpayments_df.count()
print(f"\n📊 Total order payments combined: {total_orderpayments}")

# Write to Delta table
combined_orderpayments_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SCHEMA_NAME}.OrderPayment")

print(f"✅ OrderPayment table loaded successfully!")

In [ ]:
# Summary and Verification
print("🎉 All sales tables loaded successfully!")
print("\n📅 Summary:")

# Show table counts
tables = ['Order', 'OrderLine', 'OrderPayment']

for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as count FROM {SCHEMA_NAME}.{table}").collect()[0]['count']
    print(f"   • {SCHEMA_NAME}.{table}: {count:,} records")

# Show breakdown by product line for verification
print(f"\n📦 Data Distribution Verification:")
print(f"   • Loaded from {len(PRODUCT_LINES)} product lines: {', '.join(PRODUCT_LINES)}")
print(f"   • Each product line contributed Order, OrderLine, and OrderPayment data")

print(f"\n📁 All tables are available in the '{SCHEMA_NAME}' schema")
print("🚀 Ready for cross-product analytics and reporting!")